In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import pickle



In [4]:
from sentence_transformers import CrossEncoder,InputExample
from torch.utils.data import DataLoader
from src.metric import model_evaluation



In [ ]:
path="../data/cleaned/"
# path="/content/drive/MyDrive/Project/Resume/CleanedDf/"

In [7]:
with open(path + 'train_df.pkl','rb') as f:
    train_df=pickle.load(f)
    
with open(path + 'val_df.pkl','rb') as f:
    val_df=pickle.load(f)
        
with open(path + 'test_df.pkl','rb') as f:
    test_df=pickle.load(f)
    


In [8]:
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [10]:
label_to_score = {0: 0.0, 1: 0.5, 2: 1.0}


cross_train_examples=[
    InputExample(texts=[j,r],label=float(label_to_score[l]))
    for r,j,l in zip(train_df['resume_text'],train_df['job_description_text'],train_df['label'])
]

print(f"total training example:{len(cross_train_examples)}")

total training example:5073


In [11]:
cross_train_dataloader=DataLoader(cross_train_examples,shuffle=True,batch_size=64)

In [ ]:
cross_encoder_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2',num_labels=1,device=device, max_length=384)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
model_save_path="../models/cross_encoder"
# model_save_path="/content/drive/MyDrive/Project/Resume/Models/Baseline_CrossEncoder"
os.makedirs(model_save_path,exist_ok=True)

In [ ]:
epochs=4
best_score=float('-inf')
min_delta=0.01
patience=2
count=0

for epoch in range(1,epochs+1):
    print(f"Epoch: {epoch}----------")

    cross_encoder_model.fit(train_dataloader=cross_train_dataloader,epochs=1,
                             warmup_steps=int(len(cross_train_dataloader) * epochs * 0.1),
                            show_progress_bar=True)
    
    val_pairs=list(zip(val_df['job_description_text'],val_df['resume_text']))

    scores=cross_encoder_model.predict(val_pairs,batch_size=64,show_progress_bar=False)
    
    metrics=model_evaluation(scores,val_df,'job_description_text')

    print("NDCG:",metrics['ndcg_val'])
    print("MAP:",metrics['map_score'])    

    final_score =(0.6*metrics['ndcg_val'] +
                   0.3*metrics['map_score'] +0.1*metrics['mrr_score'])
    
    if final_score>best_score+0.01:
        best_score=final_score
        cross_encoder_model.save(model_save_path)
        count=0
    else:
        count+=1
        
    if count==patience:
        print("Early Stopping")
        break
    


Epoch: 1----------


Step,Training Loss


NDCG: 0.6749039071924279
MAP: 0.7502805627831343


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch: 2----------


...

Epoch: 3----------


Step,Training Loss


NDCG: 0.6927157819763152
MAP: 0.7580448066527423
Epoch: 4----------


Step,Training Loss


NDCG: 0.6855500306497858
MAP: 0.7485482398988953
Early Stopping


In [22]:
cross_encoder_model=CrossEncoder(model_save_path,device=device)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [23]:
val_pairs=list(zip(val_df['job_description_text'],val_df['resume_text']))

scores=cross_encoder_model.predict(val_pairs,batch_size=64,show_progress_bar=False)


In [24]:
metrics=model_evaluation(scores,val_df,'job_description_text')

print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Spearman: 0.40437827145297395
Top-3 Accuracy: 1.0
NDCG: 0.6931218749615349
MAP: 0.7583671186507662
MRR: 0.8519345238095237


In [25]:
ranked_result=[]
eval_df=val_df.copy()
eval_df['score']=scores

In [26]:
print("\nScore spread within groups:")
spreads = []
for jd, group in eval_df.groupby('job_description_text'):
    if len(group) > 1:
        spreads.append(group['score'].max() - group['score'].min())

print(f"Mean spread: {np.mean(spreads):.3f}")
print(f"% groups with spread < 0.1:"f"{(np.array(spreads) < 0.1).mean():.3f}")


Score spread within groups:
Mean spread: 2.366
% groups with spread < 0.1:0.000


In [27]:
for jd,group in eval_df.groupby('job_description_text'):
    ranked_group=group.sort_values("score",ascending=False)
    ranked_result.append(ranked_group)

final_rank_df=pd.concat(ranked_result)

In [28]:
i=0
for jd,group in final_rank_df.groupby('job_description_text'):
    if(len(group)>2 and len(group)<10):
        print("Job Description:\n",jd[:300])
        print(group[['label','score']])
        i+=1
        if i==3:
            break

Job Description:
 About Chamberlain Group:
Chamberlain Group is a global leader in access solutions. Our leading brands like LiftMaster, Chamberlain, Merlin and Grifco are found in millions of homes and commercial applications across the globe. Our innovative products powered by the myQ digital ecosystem provide cust
      label     score
518       0 -0.199391
1349      0 -0.366213
1326      0 -0.394626
442       0 -0.482696
3022      0 -0.485513
939       0 -0.795424
3684      1 -0.872754
1833      0 -1.349369
Job Description:
 About Hallgate Management: Hallgate Management is a property management company with a strong commitment to providing exceptional service to our clients and residents. We pride ourselves on our dedication to excellence, integrity, and continuous growth.
Position Overview: We are seeking a detail-ori
      label     score
1309      0 -0.824239
401       0 -0.923526
300       0 -1.192520
1129      0 -1.336088
894       0 -2.050510
Job Description:
 About Us Skadd

In [29]:
test_pairs=list(zip(test_df['job_description_text'],test_df['resume_text']))

scores=cross_encoder_model.predict(test_pairs,batch_size=64,show_progress_bar=False)

metrics=model_evaluation(scores,test_df,'job_description_text')

print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])


Spearman: 0.2688762451402007
Top-3 Accuracy: 0.9642857142857143
NDCG: 0.6200074284151754
MAP: 0.716108726979836
MRR: 0.796311475409836
